# Stace labels — sanity check against NAIP z17 tiles

Each `*.geojson` in `labels/` is one labeler's set of tree-crown **MultiPolygon**
annotations (mortalitree pre/post-fire), in lon/lat (CRS84 / EPSG:4326).

**Note on structure:** a single file is *not* one z17 tile — each file holds
hundreds–thousands of crowns scattered across many z17 tiles (e.g. `agaleana`
spans ~157x60 tiles in the Sierra Nevada). The tile-metadata fields
(`zxy_val`, `x_val`, `y_val`, `url_suffix`, ...) exist in the schema but are
**all null**, so we derive the covering z17 tiles from the geometry itself.

This notebook:
1. loads a geojson and summarizes it,
2. finds the z17 tile containing the most crowns,
3. downloads that NAIP tile (USDA NAIP, pixel-aligned to the slippy grid),
4. overlays the crowns to verify alignment,
5. shows a 3x3 NAIP mosaic with crown polygons + bounding boxes (the boxes are
   what will eventually go into the CSV).

In [ ]:
import io
import math
import requests
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path
from collections import Counter

LABEL_DIR = Path("labels")

# Pick a file to inspect (small + dense is good for a first check).
GEOJSON = LABEL_DIR / "aiong05_prefire_labels.geojson"
ZOOM = 17

# Genuine USDA NAIP imagery (ArcGIS ImageServer). exportImage lets us request an
# exact Web-Mercator bbox, so we can render each z17 tile pixel-aligned to the
# slippy grid the annotators used.
NAIP_URL = ("https://gis.apfo.usda.gov/arcgis/rest/services/"
            "NAIP/USDA_CONUS_PRIME/ImageServer/exportImage")

print("Available label files:")
for p in sorted(LABEL_DIR.glob("*.geojson")):
    print(" ", p.name)

## 1. Load + inspect the geojson

In [ ]:
gdf = gpd.read_file(GEOJSON)
print(f"file: {GEOJSON.name}")
print(f"crs:  {gdf.crs}")
print(f"# features: {len(gdf)}")
print(f"geometry types: {gdf.geom_type.value_counts().to_dict()}")
print(f"property columns: {[c for c in gdf.columns if c != 'geometry']}")

minx, miny, maxx, maxy = gdf.total_bounds
print(f"lon range: [{minx:.5f}, {maxx:.5f}]")
print(f"lat range: [{miny:.5f}, {maxy:.5f}]")
gdf.head()

## 2. Slippy-tile helpers + NAIP downloader

In [ ]:
TILE = 256
R = 6378137.0
ORIGIN = math.pi * R  # 20037508.342789244  (web-mercator half-extent, meters)

def lonlat_to_tile(lon, lat, z):
    """lon/lat (deg) -> integer slippy tile (x, y) at zoom z."""
    n = 2 ** z
    x = int((lon + 180.0) / 360.0 * n)
    y = int((1.0 - math.asinh(math.tan(math.radians(lat))) / math.pi) / 2.0 * n)
    return x, y

def tile_to_merc_bounds(x, y, z):
    """slippy tile -> EPSG:3857 bounds (west, south, east, north) in meters."""
    n = 2 ** z
    west  = x / n * 2 * ORIGIN - ORIGIN
    east  = (x + 1) / n * 2 * ORIGIN - ORIGIN
    north = ORIGIN - y / n * 2 * ORIGIN
    south = ORIGIN - (y + 1) / n * 2 * ORIGIN
    return west, south, east, north

def fetch_naip(west, south, east, north, w_px, h_px):
    """Download a NAIP image for a web-mercator bbox at the given pixel size."""
    params = dict(bbox=f"{west},{south},{east},{north}", bboxSR=3857, imageSR=3857,
                  size=f"{w_px},{h_px}", format="jpgpng", f="image")
    r = requests.get(NAIP_URL, params=params, timeout=60)
    r.raise_for_status()
    return Image.open(io.BytesIO(r.content)).convert("RGB")

def fetch_naip_tile(x, y, z):
    """Download a single z/x/y NAIP tile (256x256), aligned to the slippy grid."""
    w, s, e, n = tile_to_merc_bounds(x, y, z)
    return fetch_naip(w, s, e, n, TILE, TILE)

# quick connectivity check
_t = fetch_naip_tile(*lonlat_to_tile(*gdf.geometry.iloc[0].centroid.coords[0], ZOOM), ZOOM)
print("NAIP fetch OK:", _t.size)

## 3. Find the densest z17 tile and download it

In [ ]:
# Assign each crown to a z17 tile via its lon/lat bbox center, then take the most
# populated tile. (bbox center avoids the geographic-CRS centroid warning.)
b = gdf.geometry.bounds  # minx, miny, maxx, maxy in lon/lat
tiles = [lonlat_to_tile((r.minx + r.maxx) / 2, (r.miny + r.maxy) / 2, ZOOM)
         for r in b.itertuples()]
counts = Counter(tiles)
(tx, ty), n_here = counts.most_common(1)[0]
print(f"densest z17 tile: x={tx} y={ty} z={ZOOM}  ->  {n_here} crowns")
print(f"total distinct z17 tiles touched by this file: {len(counts)}")

naip_tile = fetch_naip_tile(tx, ty, ZOOM)
naip_tile

## 4. Overlay crowns on the single NAIP tile

In [ ]:
west, south, east, north = tile_to_merc_bounds(tx, ty, ZOOM)

def merc_to_px(mx, my, west, north, w_px, h_px, span_x, span_y):
    px = (mx - west) / span_x * w_px
    py = (north - my) / span_y * h_px
    return px, py

# crowns whose centroid lands in this tile, reprojected to web-mercator
sel = gdf[[t == (tx, ty) for t in tiles]].to_crs(3857)
span_x, span_y = east - west, north - south

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(np.asarray(naip_tile), extent=[0, TILE, TILE, 0])

for geom in sel.geometry:
    polys = geom.geoms if geom.geom_type == "MultiPolygon" else [geom]
    for poly in polys:
        xs, ys = poly.exterior.xy
        px = [merc_to_px(x, y, west, north, TILE, TILE, span_x, span_y)
              for x, y in zip(xs, ys)]
        ax.add_patch(mpatches.Polygon(px, closed=True, fill=False,
                                      edgecolor="cyan", linewidth=1.2))

ax.set_xlim(0, TILE); ax.set_ylim(TILE, 0)
ax.set_title(f"{GEOJSON.name}\nNAIP z{ZOOM} tile x={tx} y={ty} — {len(sel)} crowns")
ax.axis("off")
plt.tight_layout(); plt.show()

## 5. 3x3 NAIP mosaic with crown polygons + bounding boxes

Wider view so alignment is easier to judge. The red boxes are each crown's
lon/lat bounding box — i.e. what the eventual CSV will store.

In [ ]:
PAD = 1  # tiles on each side of the dense tile -> (2*PAD+1) square mosaic
x0, x1 = tx - PAD, tx + PAD
y0, y1 = ty - PAD, ty + PAD
W = (x1 - x0 + 1) * TILE
H = (y1 - y0 + 1) * TILE

# stitch NAIP tiles into one mosaic image
mosaic = Image.new("RGB", (W, H))
for j, ytile in enumerate(range(y0, y1 + 1)):
    for i, xtile in enumerate(range(x0, x1 + 1)):
        mosaic.paste(fetch_naip_tile(xtile, ytile, ZOOM), (i * TILE, j * TILE))

# mosaic bounds in web-mercator
m_west, _, _, m_north = tile_to_merc_bounds(x0, y0, ZOOM)
_, m_south, m_east, _ = tile_to_merc_bounds(x1, y1, ZOOM)
m_span_x, m_span_y = m_east - m_west, m_north - m_south

def to_mpx(mx, my):
    return merc_to_px(mx, my, m_west, m_north, W, H, m_span_x, m_span_y)

# crowns intersecting the mosaic footprint
from shapely.geometry import box as shp_box
foot = gpd.GeoSeries([shp_box(m_west, m_south, m_east, m_north)], crs=3857).iloc[0]
gdf_m = gdf.to_crs(3857)
in_view = gdf_m[gdf_m.intersects(foot)]
print(f"{len(in_view)} crowns in the {2*PAD+1}x{2*PAD+1}-tile mosaic")

fig, ax = plt.subplots(figsize=(12, 12))
ax.imshow(np.asarray(mosaic), extent=[0, W, H, 0])
for geom in in_view.geometry:
    polys = geom.geoms if geom.geom_type == "MultiPolygon" else [geom]
    for poly in polys:
        xs, ys = poly.exterior.xy
        px = [to_mpx(x, y) for x, y in zip(xs, ys)]
        ax.add_patch(mpatches.Polygon(px, closed=True, fill=False,
                                      edgecolor="cyan", linewidth=1.0))
        bxmin, bymin, bxmax, bymax = poly.bounds
        bx0, by0 = to_mpx(bxmin, bymax)   # upper-left
        bx1, by1 = to_mpx(bxmax, bymin)   # lower-right
        ax.add_patch(mpatches.Rectangle((bx0, by0), bx1 - bx0, by1 - by0,
                                        fill=False, edgecolor="red", linewidth=0.6))

ax.set_xlim(0, W); ax.set_ylim(H, 0)
ax.set_title(f"{GEOJSON.name} — NAIP z{ZOOM} mosaic x[{x0}-{x1}] y[{y0}-{y1}]")
ax.axis("off")
plt.tight_layout(); plt.show()